In [1]:
import random
from datetime import datetime, timedelta

from faker import Faker
from pydeequ.verification import VerificationResult
from pydeequ.analyzers import *
from pydeequ.analyzers import AnalysisRunner, Size, Completeness, Maximum, Minimum, Mean, StandardDeviation, Distinctness, ApproxCountDistinct
from pydeequ.checks import Check, CheckLevel
from pydeequ.repository import FileSystemMetricsRepository, ResultKey
from pydeequ.verification import VerificationSuite
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col, to_date, lit, desc, datediff
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, DateType, TimestampType

In [2]:
def generate_users(fake: Faker, count: int):
    rows = []
    for i in range(1, count + 1):
        rows.append({
            "user_id": i,
            "email": fake.unique.email(),
            "name": fake.name(),
            "age": fake.random_int(min=10, max=100),
            "gender": random.choice(["M", "F"]),
            "job": fake.job(),
            "address": fake.address(),
            "signup": (datetime.now() - timedelta(days=random.randint(0, 365))).date(),
            "created_at": datetime.now()
        })
    return rows


user_schema = StructType([
    StructField("user_id", LongType(), False),
    StructField("email", StringType(), False),
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("job", StringType(), True),
    StructField("address", StringType(), True),
    StructField("signup", DateType(), True),
    StructField("created_at", TimestampType(), True)
])

In [3]:
spark = SparkSession.builder \
    .appName("Example Deequ") \
    .master("spark://spark-master.mmix.io:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio.mmix.io:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "mmix") \
    .config("spark.hadoop.fs.s3a.secret.key", "mmixmmix") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.shuffle.partitions", "1") \
    .getOrCreate()

26/06/14 20:23:21 WARN Utils: Your hostname, genius.local resolves to a loopback address: 127.0.0.1; using 192.168.45.182 instead (on interface en0)
26/06/14 20:23:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 20:23:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Deequ Configuration

In [4]:
repository = FileSystemMetricsRepository(spark, "s3a://mmix-prod-dataengineer-validation/deequ/sample/metrics/orders/metrics.json")
resultKey = ResultKey(spark, ResultKey.current_milli_time(), {"pipeline": "orders", "dataset": "orders", "env": "prod"})

26/06/14 20:23:23 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/06/14 20:23:35 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
users = spark.createDataFrame(data=generate_users(Faker("ko_KR"), 10), schema=user_schema)

In [7]:
expected_count = 100
email_regex = r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}$"

check = Check(spark, CheckLevel.Error, "Basic data checks") \
    .hasSize(lambda n: n == expected_count) \
    .isComplete("user_id") \
    .isComplete("email") \
    .isComplete("name") \
    .isComplete("created_at") \
    .isUnique("user_id") \
    .isUnique("email") \
    .satisfies("gender IS NULL OR gender IN ('M','F')", "gender_in_domain_or_null", lambda ratio: ratio == 1.0) \
    .satisfies("age IS NULL OR (age >= 10 AND age <= 100)", "age_range_or_null", lambda ratio: ratio == 1.0) \
    .satisfies(f"email RLIKE '{email_regex}'", "email_format", lambda ratio: ratio >= 0.99) \
    .satisfies("signup IS NULL OR signup <= current_date()", "signup_not_in_future", lambda ratio: ratio == 1.0) \
    .satisfies("signup IS NULL OR created_at >= signup", "created_at_after_signup", lambda ratio: ratio == 1.0)

warn_check = Check(spark, CheckLevel.Warning, "Users dataset warning checks") \
    .satisfies(
    """
    CASE
      WHEN gender IS NULL THEN true
      ELSE true
    END
    """,
    "noop_for_example",
    lambda _: True)

check_result = VerificationSuite(spark) \
    .onData(users) \
    .addCheck(check) \
    .addCheck(warn_check) \
    .useRepository(repository) \
    .saveOrAppendResult(resultKey) \
    .run()

check_df = VerificationResult.checkResultsAsDataFrame(spark, check_result)
check_df.show(truncate=False)

+----------------------------+-----------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+---------------------------------------------------+
|check                       |check_level|check_status|constraint                                                                                                                                         |constraint_status|constraint_message                                 |
+----------------------------+-----------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+---------------------------------------------------+
|Basic data checks           |Error      |Error       |SizeConstraint(Size(None))                                                                                                 

/opt/miniconda3/envs/enjoy-workreduce/lib/python3.10/site-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [8]:
users_enriched = users.withColumn("days_from_signup_to_created", when(col("signup").isNotNull(), datediff(to_date(col("created_at")), col("signup"))).otherwise(lit(None)))

In [9]:
analysis_result = AnalysisRunner(spark) \
    .onData(users_enriched) \
    .addAnalyzer(Size()) \
    .addAnalyzer(Completeness("user_id")) \
    .addAnalyzer(Completeness("email")) \
    .addAnalyzer(Completeness("name")) \
    .addAnalyzer(Completeness("age")) \
    .addAnalyzer(Completeness("gender")) \
    .addAnalyzer(Completeness("job")) \
    .addAnalyzer(Completeness("address")) \
    .addAnalyzer(Completeness("signup")) \
    .addAnalyzer(Completeness("created_at")) \
    .addAnalyzer(ApproxCountDistinct("user_id")) \
    .addAnalyzer(ApproxCountDistinct("email")) \
    .addAnalyzer(Distinctness("user_id")) \
    .addAnalyzer(Distinctness("email")) \
    .addAnalyzer(Minimum("age")) \
    .addAnalyzer(Maximum("age")) \
    .addAnalyzer(Mean("age")) \
    .addAnalyzer(StandardDeviation("age")) \
    .addAnalyzer(Minimum("days_from_signup_to_created")) \
    .addAnalyzer(Maximum("days_from_signup_to_created")) \
    .addAnalyzer(Mean("days_from_signup_to_created")) \
    .useRepository(repository) \
    .saveOrAppendResult(resultKey) \
    .run()

26/06/14 20:24:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [10]:
metrics_dataframe = AnalyzerContext.successMetricsAsDataFrame(spark, analysis_result)
metrics_dataframe.show(truncate=False)

+-------+---------------------------+-------------------+-----------------+
|entity |instance                   |name               |value            |
+-------+---------------------------+-------------------+-----------------+
|Column |email                      |Completeness       |1.0              |
|Column |email                      |Distinctness       |1.0              |
|Column |job                        |Completeness       |1.0              |
|Column |user_id                    |Distinctness       |1.0              |
|Column |gender                     |Completeness       |1.0              |
|Column |days_from_signup_to_created|Minimum            |1.0              |
|Column |days_from_signup_to_created|Maximum            |327.0            |
|Column |age                        |Completeness       |1.0              |
|Column |age                        |Minimum            |11.0             |
|Column |age                        |Maximum            |70.0             |
|Column |use

In [11]:
users.groupBy("gender").count().orderBy(desc("count")).show()
users.groupBy("job").count().orderBy(desc("count")).show(50, truncate=False)

+------+-----+
|gender|count|
+------+-----+
|     F|    5|
|     M|    5|
+------+-----+

+------------------------------+-----+
|job                           |count|
+------------------------------+-----+
|웹 및 멀티미디어 기획자       |1    |
|양장 및 양복 제조원           |1    |
|유치원 교사                   |1    |
|조경 기술자                   |1    |
|캐드원                        |1    |
|데이터베이스 개발자           |1    |
|영상/녹화 및 편집 기사        |1    |
|정부행정 관리자               |1    |
|가축 사육 종사원              |1    |
|기타 공학관련 기술자 및 시험원|1    |
+------------------------------+-----+



In [12]:
spark.stop()